# 05 Lineage, Tags and Discovery

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Make data <b>findable and understandable</b>: document tables and columns with comments, classify them with <b>tags</b> (for example <code>pii = email</code>), find objects with <code>information_schema</code> and search, and trace where data comes from and goes to with Unity Catalog's automatic <b>lineage</b> (in Catalog Explorer and <code>system.access.table_lineage</code>).
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
A lakehouse with thousands of undocumented tables is a swamp. Comments and tags let people (and Genie, and AI assistants) find the right table, tags drive governance such as "mask every column tagged <code>pii</code>", and lineage answers the two questions every incident starts with: "where does this number come from?" and "what breaks if I change this table?".
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Discovery features in Unity Catalog</b><br>
| Feature | What it is | How |<br>|---|---|---|<br>| <b>Comments</b> | Free-text descriptions on catalogs, schemas, tables, columns, volumes, functions | <code>COMMENT ON</code>, <code>ALTER COLUMN ... COMMENT</code>, or AI-generated suggestions in Catalog Explorer |<br>| <b>Tags</b> | Key-value labels on catalogs, schemas, tables, columns, volumes | <code>ALTER ... SET TAGS ('k' = 'v')</code>, <code>UNSET TAGS</code> |<br>| <b>Governed tags</b> | Account-level tag definitions with allowed values and permissions on who may assign them | Created by admins |<br>| <b>Lineage</b> | Captured automatically for queries run on Unity Catalog compute, down to column level | Catalog Explorer <b>Lineage</b> tab, <code>system.access.table_lineage</code>, <code>system.access.column_lineage</code> |<br>| <b>Search and metadata</b> | Search bar, Catalog Explorer, <code>information_schema</code> | UI and SQL |<br><br>
Lineage is captured from what actually runs (notebooks, jobs, pipelines, SQL queries, dashboards), so there's nothing to maintain by hand.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
Finance reported that "net revenue" in a dashboard was 4% lower than in their spreadsheet. With the lineage graph, the analytics engineer followed the dashboard's dataset back through <code>gold.revenue_daily</code> and <code>silver.orders</code> to a filter in Silver that excluded orders with status <code>partially_refunded</code>. That took 10 minutes instead of a day of reading code, and the Silver table's column comments now say exactly what's excluded.
</div>

## Syntax

```sql
COMMENT ON TABLE c.s.orders IS 'One row per order, Silver layer';
ALTER TABLE c.s.orders ALTER COLUMN amount COMMENT 'Order total in EUR, incl. VAT';
ALTER TABLE c.s.orders SET TAGS ('domain' = 'sales', 'layer' = 'silver');
ALTER TABLE c.s.customers ALTER COLUMN email SET TAGS ('pii' = 'email');
ALTER TABLE c.s.orders UNSET TAGS ('layer');
SELECT * FROM c.information_schema.column_tags WHERE tag_name = 'pii';
SELECT * FROM system.access.table_lineage WHERE target_table_full_name = 'c.s.orders_gold';
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds a tiny Bronze → Silver → Gold pipeline in the <code>governance</code> schema, so there is lineage and something to document.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Three tables. The Gold table shows revenue per country.
</div>

In [0]:
CATALOG = "workspace"
DB = f"{CATALOG}.governance"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {DB}.lineage_orders_bronze AS
    SELECT * FROM VALUES (1, 'asha@example.com', 'IN', '120.50'), (2, 'ben@example.com', 'UK', '80.00'),
                         (3, 'chen@example.com', 'AE', 'n/a'),   (4, 'asha@example.com', 'IN', '35.25')
    AS t(order_id, email, country, amount_raw)
""")
spark.sql(f"""
    CREATE OR REPLACE TABLE {DB}.lineage_orders_silver AS
    SELECT order_id, email, country, try_cast(amount_raw AS DECIMAL(10, 2)) AS amount
    FROM {DB}.lineage_orders_bronze
    WHERE try_cast(amount_raw AS DECIMAL(10, 2)) IS NOT NULL
""")
spark.sql(f"""
    CREATE OR REPLACE TABLE {DB}.lineage_revenue_gold AS
    SELECT country, SUM(amount) AS revenue, COUNT(*) AS orders
    FROM {DB}.lineage_orders_silver GROUP BY country
""")
spark.table(f"{DB}.lineage_revenue_gold").orderBy("country").show()

## 1. Document tables and columns with comments

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds a table comment and column comments to the Silver table, then reads them back with <code>DESCRIBE</code>.<br><br>
<b>Why it matters</b><br>Comments are shown in Catalog Explorer, in search results and to Genie and the Databricks Assistant, which use them to write better queries. Document what a row means, units, and anything that's filtered out.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The table comment, and comments next to <code>email</code> and <code>amount</code>.
</div>

In [0]:
spark.sql(f"COMMENT ON TABLE {DB}.lineage_orders_silver IS 'Silver orders: one row per valid order. Rows with non-numeric amounts are dropped.'")
spark.sql(f"ALTER TABLE {DB}.lineage_orders_silver ALTER COLUMN email COMMENT 'Customer email address (PII)'")
spark.sql(f"ALTER TABLE {DB}.lineage_orders_silver ALTER COLUMN amount COMMENT 'Order total in EUR including VAT'")

spark.sql(f"DESCRIBE TABLE {DB}.lineage_orders_silver").show(truncate=False)
print(spark.sql(f"DESCRIBE TABLE EXTENDED {DB}.lineage_orders_silver").filter("col_name = 'Comment'").first()["data_type"])

## 2. Classify with tags

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Tags the Silver table with its domain and layer, and tags the <code>email</code> column as PII.<br><br>
<b>Why it matters</b><br>Tags are structured, so they can be queried and used by policies: "list all PII columns", "which tables belong to finance?". Comments explain, tags classify.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Tags are a Unity Catalog feature. If your workspace doesn't allow setting them, the cell prints a message.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No error, or a message if tags aren't available.
</div>

In [0]:
try:
    spark.sql(f"ALTER TABLE {DB}.lineage_orders_silver SET TAGS ('domain' = 'sales', 'layer' = 'silver')")
    spark.sql(f"ALTER TABLE {DB}.lineage_orders_silver ALTER COLUMN email SET TAGS ('pii' = 'email')")
    spark.sql(f"ALTER TABLE {DB}.lineage_orders_bronze ALTER COLUMN email SET TAGS ('pii' = 'email')")
    print("tags set")
except Exception as e:
    print("Tags not available here:", type(e).__name__, str(e)[:200])

## 3. Query tags and comments with `information_schema`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists table tags and column tags in the <code>governance</code> schema, and every column that has a comment.<br><br>
<b>Why it matters</b><br>This turns metadata into a report: a PII inventory, or a documentation coverage check ("which Gold columns have no comment?").
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The <code>domain</code> and <code>layer</code> tags on the Silver table, <code>pii = email</code> on two <code>email</code> columns, and the commented columns.
</div>

In [0]:
for view, cols in [("table_tags", "table_name, tag_name, tag_value"),
                   ("column_tags", "table_name, column_name, tag_name, tag_value")]:
    try:
        display(spark.sql(f"SELECT {cols} FROM {CATALOG}.information_schema.{view} WHERE schema_name = 'governance' ORDER BY ALL"))
    except Exception as e:
        print(f"{view} not available here:", type(e).__name__)

display(spark.sql(f"""
    SELECT table_name, column_name, comment
    FROM {CATALOG}.information_schema.columns
    WHERE table_schema = 'governance' AND comment IS NOT NULL
"""))

## 4. Documentation coverage check

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Computes, for each table in the schema, how many columns have comments, using the catalog API so it also works in plain Spark.<br><br>
<b>Why it matters</b><br>Simple metrics like "% of Gold columns documented" make documentation part of the definition of done.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The Silver table has 2 of 4 columns documented. The others have none.
</div>

In [0]:
def doc_coverage(schema_name):
    rows = []
    for t in spark.catalog.listTables(schema_name):
        cols = spark.catalog.listColumns(f"{schema_name}.{t.name}")
        documented = sum(1 for c in cols if c.description)
        rows.append((t.name, documented, len(cols)))
    return rows

for name, documented, total in sorted(doc_coverage(DB)):
    if name.startswith("lineage_"):
        print(f"{name:<25} {documented}/{total} columns documented")

## 5. Lineage in Catalog Explorer

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Describes how to see lineage in the UI. There's no code to run.<br><br>
<b>Why it matters</b><br>The lineage graph is the fastest way to answer "where does this come from?" and "who uses this?" for any table, including the notebooks, jobs and dashboards involved.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
After following the steps, a graph <code>lineage_orders_bronze → lineage_orders_silver → lineage_revenue_gold</code>.
</div>

**Steps**

1. Open **Catalog** in the left sidebar
2. Go to `workspace` → `governance` → `lineage_revenue_gold`
3. Open the **Lineage** tab and click **See lineage graph**
4. Click the `+` on a node to expand upstream or downstream. Click a column to see **column-level lineage** (for example `revenue` comes from `amount_raw`)
5. The lineage panel also lists the notebooks, jobs and queries that read or wrote the table

Lineage can take a few minutes to appear after a query runs.

## 6. Lineage with system tables

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Queries <code>system.access.table_lineage</code> for the upstream sources of the Gold table, and <code>system.access.column_lineage</code> for the <code>revenue</code> column.<br><br>
<b>Why it matters</b><br>Lineage in SQL can be automated: impact analysis before a schema change ("list every downstream table of <code>silver.orders</code>"), or audits.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
System tables may not be enabled or accessible in Free Edition. The cell prints a message if so. Records can take some time to appear.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
On workspaces with lineage system tables: rows showing Bronze → Silver and Silver → Gold. Otherwise, a message.
</div>

In [0]:
try:
    display(spark.sql(f"""
        SELECT source_table_full_name, target_table_full_name, entity_type, MAX(event_time) AS last_seen
        FROM system.access.table_lineage
        WHERE target_table_full_name LIKE '{CATALOG}.governance.lineage_%'
        GROUP BY ALL ORDER BY last_seen DESC
    """))
    display(spark.sql(f"""
        SELECT DISTINCT source_table_full_name, source_column_name, target_column_name
        FROM system.access.column_lineage
        WHERE target_table_full_name = '{CATALOG}.governance.lineage_revenue_gold'
    """))
except Exception as e:
    print("Lineage system tables not available here:", type(e).__name__)

## Under the hood

```
query runs on Unity Catalog compute ──▶ UC records reads + writes (tables and columns) ──▶ lineage service
                                                                                         ├─ Catalog Explorer graph
                                                                                         └─ system.access.table_lineage / column_lineage
COMMENT / SET TAGS ──▶ stored as metadata in UC ──▶ information_schema, search, Genie, Assistant
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata</span> comments, tags and lineage never touch the data files.

In [0]:
spark.sql(f"DESCRIBE TABLE EXTENDED {DB}.lineage_orders_silver").filter("col_name IN ('Comment', 'Owner', 'Type')").show(truncate=False)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: no lineage for a table</b><br>
Lineage is captured for queries on Unity Catalog compute. Writes through paths instead of table names, or from external tools, may not appear. Wait a few minutes after the query.<br><br>
<b>⛔ Problem: comments that repeat the column name</b><br><code>customer_id: the customer id</code> helps nobody. Explain meaning, units, filters and edge cases.<br><br>
**⛔ Problem: free-text tags (<code>PII</code>, <code>pii</code>, <code>Pii</code>)**<br>Agree on keys and values, or use governed tags with allowed values.<br><br>
<b>⛔ Problem: documenting in a wiki instead of the catalog</b><br>Wikis drift. Comments and tags live with the data and are visible where people query.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How does Unity Catalog capture lineage?</b><br>
Automatically, for queries run on Unity Catalog-enabled compute: it records which tables and columns were read and written by notebooks, jobs, pipelines, SQL queries and dashboards. It's shown in Catalog Explorer and exposed in <code>system.access.table_lineage</code> and <code>system.access.column_lineage</code>.<br><br>

<b>🎤 2. How would you do impact analysis before changing a Silver table?</b><br>
Look at the downstream lineage in Catalog Explorer, or query <code>system.access.table_lineage</code> (and <code>column_lineage</code> for specific columns) for targets whose source is the Silver table, then check the listed jobs, dashboards and owners.<br><br>

<b>🎤 3. What is the difference between comments and tags?</b><br>
Comments are free-text documentation for people and AI tools. Tags are structured key-value labels used for classification, search and policies, for example tagging columns with <code>pii = email</code>.<br><br>

<b>🎤 4. How would you find all PII columns in a catalog?</b><br>
Query <code>information_schema.column_tags</code> for the PII tag, and as a safety net search <code>information_schema.columns</code> for suspicious column names.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A data engineer wants to know which downstream tables and dashboards use <code>prod.silver.orders</code> before changing a column. What is the best tool?</b><br>
A. <code>DESCRIBE HISTORY prod.silver.orders</code><br>
B. Unity Catalog lineage (Catalog Explorer lineage graph or lineage system tables)<br>
C. <code>SHOW GRANTS ON TABLE prod.silver.orders</code><br>
D. The Spark UI<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. Which command labels the <code>email</code> column of a table as personal data in a way that can be queried later?</b><br>
A. <code>ALTER TABLE t ALTER COLUMN email SET TAGS ('pii' = 'email')</code><br>
B. <code>ALTER TABLE t SET TBLPROPERTIES ('email' = 'pii')</code><br>
C. <code>COMMENT ON COLUMN t.email IS 'pii'</code> only<br>
D. <code>GRANT PII ON COLUMN t.email</code><br>
<details><summary><b>Show answer</b></summary><b>A.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Add comments to every column of <code>lineage_revenue_gold</code> and to the table itself, then run <code>doc_coverage</code> again</li><li>(Databricks) Tag the Gold table with <code>domain = sales</code> and <code>layer = gold</code>, then list all tables in the schema that have a <code>layer</code> tag</li><li>(Databricks) Find all downstream tables of <code>lineage_orders_bronze</code> using <code>system.access.table_lineage</code></li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Document the Gold table fully
gold = f"{DB}.lineage_revenue_gold"
spark.sql(f"COMMENT ON TABLE {gold} IS 'Gold: revenue and order count per country, from valid Silver orders'")
for col, text in [("country", "ISO country code of the customer"), ("revenue", "Sum of order amounts in EUR incl. VAT"),
                  ("orders", "Number of valid orders")]:
    spark.sql(f"ALTER TABLE {gold} ALTER COLUMN {col} COMMENT '{text}'")
for name, documented, total in sorted(doc_coverage(DB)):
    if name == "lineage_revenue_gold":
        print(f"{name}: {documented}/{total} columns documented")

In [0]:
# 2. Tags and a query over them
try:
    spark.sql(f"ALTER TABLE {gold} SET TAGS ('domain' = 'sales', 'layer' = 'gold')")
    display(spark.sql(f"SELECT table_name, tag_value AS layer FROM {CATALOG}.information_schema.table_tags WHERE schema_name = 'governance' AND tag_name = 'layer'"))
except Exception as e:
    print("Tags not available here:", type(e).__name__)

# 3. Downstream of Bronze
try:
    display(spark.sql(f"""
        SELECT DISTINCT target_table_full_name FROM system.access.table_lineage
        WHERE source_table_full_name = '{CATALOG}.governance.lineage_orders_bronze' AND target_table_full_name IS NOT NULL
    """))
except Exception as e:
    print("Lineage system tables not available here:", type(e).__name__)

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Comments explain (tables, columns, schemas, volumes). Tags classify (<code>pii</code>, <code>domain</code>, <code>layer</code>)</li><li><code>information_schema.table_tags</code> / <code>column_tags</code> / <code>columns</code> turn metadata into reports</li><li>Lineage is automatic for Unity Catalog compute: Catalog Explorer graph, <code>system.access.table_lineage</code>, <code>column_lineage</code></li><li>Use lineage for root-cause analysis (upstream) and impact analysis (downstream)</li><li>Good metadata also makes Genie and the Assistant more accurate</li></ul>
</div>

| Task | Command |
|---|---|
| Table comment | `COMMENT ON TABLE t IS '...'` |
| Column comment | `ALTER TABLE t ALTER COLUMN c COMMENT '...'` |
| Tag table / column | `ALTER TABLE t SET TAGS ('k'='v')`, `ALTER COLUMN c SET TAGS (...)` |
| Remove tag | `ALTER TABLE t UNSET TAGS ('k')` |
| Find tags | `information_schema.table_tags`, `column_tags` |
| Lineage | Catalog Explorer → Lineage, `system.access.table_lineage` |

## Git commit

```
git add 06_unity_catalog/05_lineage_tags_discovery.ipynb
git commit -m "add 06_05 lineage tags and discovery notebook"
```